# 01 - Couche Bronze : Ingestion des données Open Data Enedis

Ce notebook télécharge les données brutes depuis l'API Open Data Enedis
et les stocke au format Delta Lake, sans aucune transformation.

## Configuration générale

In [0]:
# Importation des bibliothèques nécessaires

import requests          # Bibliothèque HTTP pour appeler des APIs et télécharger des fichiers
import pandas as pd      
from io import StringIO  # Outil pour traiter des chaînes de caractères comme des fichiers


# Importation des fonctions PySpark disponibles dans le module SQL
from pyspark.sql.functions import (
    current_timestamp,  
    current_date,       
    lit                 # Crée une colonne avec une valeur constante (littérale)
)

from datetime import datetime  

In [0]:
# --- Définition des jeux de données à ingérer depuis l'API Open Data Enedis ---
# Chaque entrée est un dictionnaire contenant l'URL de téléchargement et une description lisible

DATASETS = {
    "conso_inf36_region": {
        
        "url": "https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv", # URL de l'API Enedis pour le jeu de données de consommation résidentielle inférieure à 36 kVA par région. Le format CSV est demandé via le chemin /exports/csv
        
        "description": "Consommation résidentielle inférieure à 36 kVA par région (annuelle)"
    },
    "conso_sup36_region": {
        
        "url": "https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv", # Même logique pour la consommation professionnelle supérieure à 36 kVA
        
        "description": "Consommation professionnelle supérieure à 36 kVA par région (annuelle)"
    }
}

# Affichage de confirmation dans les logs du notebook
print("Configuration chargée")
print(f"Datasets à ingérer : {list(DATASETS.keys())}")

Configuration chargée
Datasets à ingérer : ['conso_inf36_region', 'conso_sup36_region']


## Fonction d'ingestion générique

In [0]:
def ingest_dataset(dataset_name: str, dataset_config: dict) -> dict:

    ingestion_start = datetime.now()

    print(f"Ingestion en cours : {dataset_name}")
    print(f"  Source URL : {dataset_config['url']}")


    # --- Étape 1 : Téléchargement des données depuis l'API ---

    params = {
        "limit": 15000,   # 15 000 lignes pour vérifier que le pipeline fonctionne correctement. Une fois les tests validés, on repassera à -1 pour récupérer l'ensemble des lignes sans aucune limite.
        "delimiter": ";"
    }

    response = requests.get(
        dataset_config["url"],
        params=params,
        timeout=120       # On attend au maximum 120 secondes avant d'abandonner la requête
    )

    response.raise_for_status()  # Lève une exception automatiquement si le code HTTP indique une erreur (4xx, 5xx). Cela interrompt la fonction proprement si l'API ne répond pas correctement.


    # --- Étape 2 : Conversion du contenu CSV en DataFrame Spark ---

    pdf = pd.read_csv(StringIO(response.text), sep=";")  # response.text contient le contenu brut de la réponse. StringIO transforme cette chaîne en fichier virtuel que pandas peut lire directement.

    print(f"  Lignes reçues : {len(pdf):,}")
    print(f"  Colonnes      : {list(pdf.columns)}")

    sdf = spark.createDataFrame(pdf)  # Convertit le DataFrame Pandas en DataFrame Spark, ce qui permet de traiter des volumes bien plus importants et d'écrire en Delta Lake.


    # --- Étape 3 : Ajout des métadonnées d'ingestion ---
    # Ces métadonnées permettront de tracer la lignée de la donnée (origine et date d'arrivée).

    sdf = (
        sdf
        .withColumn("_ingestion_timestamp", current_timestamp())   # Horodatage exact de l'ingestion

        .withColumn("_ingestion_date", current_date())             # Date du jour de l'ingestion

        .withColumn("_source_dataset", lit(dataset_name))         # Nom du jeu de données source. lit() crée une colonne avec la même valeur constante pour chaque ligne.

        .withColumn("_source_url", lit(dataset_config["url"]))    # URL exacte utilisée pour le téléchargement (traçabilité complète)

        .withColumn("_source_description", lit(dataset_config["description"]))  # Description lisible du jeu de données
    )

    # --- Étape 4 : Écriture en table Delta Lake managée (couche Bronze) ---

    table_name = f"bronze_{dataset_name}"

    (
        sdf.write
           .format("delta")                      # Format Delta Lake : ajoute les transactions ACID et le versioning
           .mode("overwrite")                    # Si la table existe déjà, on la remplace intégralement
           .option("overwriteSchema", "true")    # En cas de changement de schéma, on l'accepte automatiquement
           .saveAsTable(table_name)
    )

    duration_s = (datetime.now() - ingestion_start).seconds

    print(f"  Succès en {duration_s}s  ->  table : {table_name}")

    return {                                          # Retour d'un dictionnaire de métadonnées pour alimenter le rapport d'ingestion
        "dataset":          dataset_name,          
        "table":            table_name,              
        "rows":             len(pdf),                
        "columns":          len(pdf.columns),      
        "duration_seconds": duration_s,              
        "status":           "SUCCESS",               
        "timestamp":        ingestion_start.isoformat()  # Horodatage ISO 8601 du début de l'ingestion
    }

## Lancement de l'ingestion

In [0]:
# Liste qui va accumuler les metadonnées de chaque ingestion (succes ou echec). Cela permet de générer un rapport consolidé à la fin

ingestion_log = []

for ds_name, ds_config in DATASETS.items():
    try:
        
        meta = ingest_dataset(ds_name, ds_config)       # Appel de la fonction d'ingestion pour ce dataset
       
        ingestion_log.append(meta)                      # Ajout des metadonnées de succes dans la liste de logs

    except Exception as exc:                            # En cas d'erreur (reseau, API indisponible, données malformées...), on ne fait pas planter tout le pipeline : on logue l'échec et on passe au suivant
        print(f"Erreur pour {ds_name} : {exc}")

        ingestion_log.append({                                  # Ajout des metadonnées d'échec dans la liste de logs
            "dataset":   ds_name,
            "status":    "FAILED",
            "error":     str(exc),                               # Message d'erreur converti en chaine de caracteres
            "timestamp": datetime.now().isoformat()
        })


# --- Affichage du rapport d'ingestion consolidé ---


print("\n" + "="*60)
print("RAPPORT D'INGESTION - COUCHE BRONZE")
print("="*60)

for log in ingestion_log:                                       # Récupération du nombre de lignes avec valeur par défaut "N/A" si l'ingestion a échoué
    
    rows = f"{log.get('rows', 'N/A'):,}" if isinstance(log.get('rows'), int) else "N/A"
    secs = log.get('duration_seconds', '-')
    print(f"  [{log['status']}]  {log['dataset']:<35} | {rows:>10} lignes | {secs}s")

Ingestion en cours : conso_inf36_region
  Source URL : https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv
  Lignes reçues : 15,000
  Colonnes      : ['horodate', 'region', 'code_region', 'profil', 'plage_de_puissance_souscrite', 'nb_points_soutirage', 'total_energie_soutiree_wh', 'courbe_moyenne_ndeg1_wh', 'indice_representativite_courbe_ndeg1', 'courbe_moyenne_ndeg2_wh', 'indice_representativite_courbe_ndeg2', 'courbe_moyenne_ndeg1_ndeg2_wh', 'indice_representativite_courbe_ndeg1_ndeg2', 'jour_max_du_mois_0_1', 'semaine_max_du_mois_0_1']
  Succès en 49s  ->  table : bronze_conso_inf36_region
Ingestion en cours : conso_sup36_region
  Source URL : https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv
  Lignes reçues : 15,000
  Colonnes      : ['horodate', 'region', 'code_region', 'profil', 'plage_de_puissance_souscrite', 'secteur_activite', 'nb_points_soutirage', 'total_energie_soutiree_wh', 'courbe_moyenne_n

## Vérification - Aperçu des données Bronze

In [0]:
# On relit chaque table depuis le catalogue pour confirmer que les données ont bien été persistées

for ds_name in DATASETS.keys():

    table_name = f"bronze_{ds_name}"

    try:
        df = spark.read.table(table_name)

        print(f"\nTable : {table_name}")
        print(f"  Lignes   : {df.count():,}")                        # count() déclenche une action Spark (parcourt toutes les partitions)
        print(f"  Colonnes : {df.columns}")

        df.limit(5).display()

    except Exception as exc:                                        # Si la lecture échoue (table inexistante, ingestion ratée precedemment...)

        print(f"Impossible de lire {table_name} : {exc}")


Table : bronze_conso_inf36_region
  Lignes   : 15,000
  Colonnes : ['horodate', 'region', 'code_region', 'profil', 'plage_de_puissance_souscrite', 'nb_points_soutirage', 'total_energie_soutiree_wh', 'courbe_moyenne_ndeg1_wh', 'indice_representativite_courbe_ndeg1', 'courbe_moyenne_ndeg2_wh', 'indice_representativite_courbe_ndeg2', 'courbe_moyenne_ndeg1_ndeg2_wh', 'indice_representativite_courbe_ndeg1_ndeg2', 'jour_max_du_mois_0_1', 'semaine_max_du_mois_0_1', '_ingestion_timestamp', '_ingestion_date', '_source_dataset', '_source_url', '_source_description']


horodate,region,code_region,profil,plage_de_puissance_souscrite,nb_points_soutirage,total_energie_soutiree_wh,courbe_moyenne_ndeg1_wh,indice_representativite_courbe_ndeg1,courbe_moyenne_ndeg2_wh,indice_representativite_courbe_ndeg2,courbe_moyenne_ndeg1_ndeg2_wh,indice_representativite_courbe_ndeg1_ndeg2,jour_max_du_mois_0_1,semaine_max_du_mois_0_1,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_url,_source_description
2025-06-30T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,73695,1.6409232E7,402.0,22,247.0,22,324.0,45,1,1,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)
2024-09-30T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,73915,1.7178965E7,479.0,21,215.0,21,348.0,43,0,0,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)
2024-03-31T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,61820,1.9123005E7,515.0,22,266.0,22,392.0,45,0,0,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)
2023-06-30T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,20528,4713779.0,227.0,22,206.0,21,311.0,43,0,0,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)
2024-06-30T22:00:00+00:00,Île-de-France,11,RES3,P1: ]0-9] kVA,73664,1.4114932E7,400.0,21,188.0,21,294.0,42,0,0,2026-04-29T15:39:00.117Z,2026-04-29,conso_inf36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-inf36-region/exports/csv,Consommation résidentielle inférieure à 36 kVA par région (annuelle)



Table : bronze_conso_sup36_region
  Lignes   : 15,000
  Colonnes : ['horodate', 'region', 'code_region', 'profil', 'plage_de_puissance_souscrite', 'secteur_activite', 'nb_points_soutirage', 'total_energie_soutiree_wh', 'courbe_moyenne_ndeg1_wh', 'indice_representativite_courbe_ndeg1', 'courbe_moyenne_ndeg2_wh', 'indice_representativite_courbe_ndeg2', 'courbe_moyenne_ndeg1_ndeg2_wh', 'indice_representativite_courbe_ndeg1_ndeg2', 'jour_max_du_mois_0_1', 'semaine_max_du_mois_0_1', '_ingestion_timestamp', '_ingestion_date', '_source_dataset', '_source_url', '_source_description']


horodate,region,code_region,profil,plage_de_puissance_souscrite,secteur_activite,nb_points_soutirage,total_energie_soutiree_wh,courbe_moyenne_ndeg1_wh,indice_representativite_courbe_ndeg1,courbe_moyenne_ndeg2_wh,indice_representativite_courbe_ndeg2,courbe_moyenne_ndeg1_ndeg2_wh,indice_representativite_courbe_ndeg1_ndeg2,jour_max_du_mois_0_1,semaine_max_du_mois_0_1,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_url,_source_description
2024-09-30T23:00:00+00:00,Normandie,28,ENT1 (+ ENT2),P4: ]250-1000] kVA,S2: Industrie,5,112666.0,36944.0,50,null,S,22533.0,100,0,0,2026-04-29T15:39:18.007Z,2026-04-29,conso_sup36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv,Consommation professionnelle supérieure à 36 kVA par région (annuelle)
2023-09-30T22:30:00+00:00,Provence-Alpes-Côte d'Azur,93,ENT1 (+ ENT2),P7: Total > 250 kVA,S1: Agriculture,1,null,null,S,null,S,null,S,0,0,2026-04-29T15:39:18.007Z,2026-04-29,conso_sup36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv,Consommation professionnelle supérieure à 36 kVA par région (annuelle)
2023-12-31T23:30:00+00:00,Provence-Alpes-Côte d'Azur,93,ENT1 (+ ENT2),P7: Total > 250 kVA,S1: Agriculture,1,null,null,S,null,S,null,S,0,0,2026-04-29T15:39:18.007Z,2026-04-29,conso_sup36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv,Consommation professionnelle supérieure à 36 kVA par région (annuelle)
2024-03-31T22:30:00+00:00,Provence-Alpes-Côte d'Azur,93,ENT1 (+ ENT2),P7: Total > 250 kVA,S1: Agriculture,1,null,null,S,null,S,null,S,0,0,2026-04-29T15:39:18.007Z,2026-04-29,conso_sup36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv,Consommation professionnelle supérieure à 36 kVA par région (annuelle)
2024-12-31T23:30:00+00:00,Provence-Alpes-Côte d'Azur,93,ENT1 (+ ENT2),P7: Total > 250 kVA,S1: Agriculture,1,null,null,S,null,S,null,S,0,0,2026-04-29T15:39:18.007Z,2026-04-29,conso_sup36_region,https://opendata.enedis.fr/api/explore/v2.1/catalog/datasets/conso-sup36-region/exports/csv,Consommation professionnelle supérieure à 36 kVA par région (annuelle)


## Récapitulatif de la couche Bronze

| Zone   | Format      | Transformation | Stockage          | Metadonnees ajoutees |
|--------|-------------|----------------|-------------------|----------------------|
| Bronze | Delta Lake  | Aucune (raw)   | Table managee     | _ingestion_timestamp, _ingestion_date, _source_dataset, _source_url |

Tables créées : bronze_conso_inf36_region, bronze_conso_sup36_region



Nous arrivons ainsi à la fin du step Bronze. Passons maintenant au notebook 02_Silver_Transformation pour nettoyer et enrichir ces données.